# Benchmarking LLMs using Sand-Bob

You can use Sand-Bob to benchmark different language models solving coding-tasks. In this notebook we demonstrate how. Before starting, we define the task and the correct result.

In [1]:
prompt = "Count the number of b's in blueberry."
ground_truth = 2

Next, we determine which LLMs are available on the server.

In [2]:
import openai
client = openai.OpenAI(base_url="http://localhost:11434/v1",
                       api_key="none"
                      )

print("\n".join([model.id for model in client.models.list().data]))

gemma3:4b
mistral:latest
embeddinggemma:latest
mxbai-embed-large:latest
qwen:0.5b
phi3:instruct
phi3:latest
mistral-nemo:latest
llama3.1:latest


In [3]:
llms_to_test = ["mistral", "qwen:0.5b"]

In [4]:
import numpy as np
from sand_bob import generate_code, config_llms

model_performance = {}
for model in llms_to_test:
    config_llms(model=model)

    results = generate_code(prompt,
                  n_parallel=3,
                  n_iterative=2,
                  n_codefix_attempts=2,
                  n_feedback_iterations=0, # do not use a vision-model
                  final_touch=False, # do not beautify the final notebook, we just need the result
    )

    print(model, "results:", [str(r.final_result)[:10] for r in results])

    correct = 0
    for r in results:
        try:
            if int(r.final_result) == ground_truth:
                correct += 1
        except:
            pass

    model_performance[model] = correct / len(results)
    print("Success rate:", model_performance[model])

mistral results: ['None', "{'result':", "{'blueberr", 'None', "{'result':", '2']
Success rate: 0.16666666666666666


qwen:0.5b results: ['None', 'None', 'None', 'None', 'None', 'None']
Success rate: 0.0


The model which performed best was:

In [5]:
max(model_performance, key=model_performance.get)

'mistral'